# 02 — Baseline Skenario A (Proyek TTKI P1)

**Skenario A** adalah titik acuan: pembagian data **acak berstrata**, memakai fitur **F-URL + F-Page** yang lolos audit
(46 fitur; 4 fitur "belum jelas" dikeluarkan). Dua model dipakai sebagai pembanding umum, bukan untuk mencari model terbaik:
Logistic Regression (linear) dan Random Forest (ensemble pohon).

**Aturan yang dijaga di notebook ini**
- Imputasi dan scaling dipelajari hanya dari data latih di setiap fold (dibungkus dalam `Pipeline`).
- Semua parameter model tetap sesuai proposal; tidak ada tuning.
- **Test final tidak disentuh.** Sel evaluasi test final ada di bagian akhir dan dimatikan (`RUN_FINAL_TEST = False`)
  sampai rancangan dibekukan. Hasil di notebook ini hanya dari cross-validation pada development set.

In [1]:
import os, json, time, platform
import numpy as np, pandas as pd, sklearn
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, roc_auc_score, precision_score, recall_score, confusion_matrix

CFG_CANDIDATES = ["../configs/experiment_config.json", "configs/experiment_config.json"]
CFG_PATH = next(p for p in CFG_CANDIDATES if os.path.exists(p))
cfg = json.load(open(CFG_PATH))
ROOT = os.path.dirname(os.path.dirname(CFG_PATH)) or "."
RESULTS_DIR = os.path.join(ROOT, "results"); os.makedirs(RESULTS_DIR, exist_ok=True)

DATA_CANDIDATES = [os.path.join(ROOT, "data", cfg["dataset"]["file"]),
                   "/mnt/user-data/uploads/" + cfg["dataset"]["file"]]
DATA_PATH = next(p for p in DATA_CANDIDATES if os.path.exists(p))
SEED = cfg["seed"]
RUN_FINAL_TEST = False          # jangan diaktifkan sebelum rancangan dibekukan

print("Python", platform.python_version(), "| sklearn", sklearn.__version__, "| pandas", pd.__version__)
print("Data:", DATA_PATH, "| seed:", SEED)

Python 3.12.3 | sklearn 1.9.1 | pandas 3.1.1
Data: ../data/PhiUSIIL_Phishing_URL_Dataset.csv | seed: 42


## 1. Data, label, dan fitur skenario A
Label dibalik supaya phishing menjadi kelas positif (`y = 1`). Fitur skenario A diambil dari konfigurasi,
yaitu F-URL ditambah F-Page, tanpa empat fitur yang asal pembentukannya belum jelas.

In [2]:
df = pd.read_csv(DATA_PATH)
y = (1 - df["label"]).values
FEATURES_A = cfg["features"]["F_URL"] + cfg["features"]["F_Page"]
assert not set(FEATURES_A) & set(cfg["features"]["excluded_unclear"])
X = df[FEATURES_A].astype(float)
print("Fitur skenario A:", len(FEATURES_A), "| baris:", len(X), "| phishing:", int(y.sum()), f"({y.mean():.1%})")

Fitur skenario A: 46 | baris: 235795 | phishing: 100945 (42.8%)


## 2. Pembagian data (acak berstrata)
Sekitar 20% data disisihkan sebagai **test final** dan baru dipakai di akhir. Sisanya (development set) dipakai untuk
5-fold cross-validation. Indeks split disimpan supaya skenario B memakai split yang sama persis dengan A.

In [4]:
idx = np.arange(len(df))
dev, test = next(StratifiedKFold(5, shuffle=True, random_state=SEED).split(idx, y))
folds = list(StratifiedKFold(5, shuffle=True, random_state=SEED).split(dev, y[dev]))   # indeks relatif terhadap dev

np.savez(os.path.join(RESULTS_DIR, "split_indices_random.npz"),
         dev=dev, test=test,
         **{f"fold{i+1}_train": dev[tr] for i,(tr,va) in enumerate(folds)},
         **{f"fold{i+1}_val":   dev[va] for i,(tr,va) in enumerate(folds)})
print("development:", len(dev), "| test final:", len(test), f"({len(test)/len(df):.1%})")
print("rasio phishing dev/test:", round(y[dev].mean(),4), "/", round(y[test].mean(),4))

development: 188636 | test final: 47159 (20.0%)
rasio phishing dev/test: 0.4281 / 0.4281


## 3. Model dan pipeline
Konfigurasi sesuai proposal. Pipeline membungkus imputasi (median) dan, khusus Logistic Regression, scaling.
Karena semuanya ada di dalam `Pipeline`, setiap fold mempelajari median dan skala hanya dari data latih fold itu.

In [5]:
def make_pipeline(name):
    if name == "logistic_regression":
        p = cfg["models"][name]
        return Pipeline([("impute", SimpleImputer(strategy="median")),
                         ("scale", StandardScaler()),
                         ("clf", LogisticRegression(penalty=p["penalty"], C=p["C"], solver=p["solver"],
                                                    max_iter=p["max_iter"]))])
    p = cfg["models"][name]
    return Pipeline([("impute", SimpleImputer(strategy="median")),
                     ("clf", RandomForestClassifier(n_estimators=p["n_estimators"], max_depth=p["max_depth"],
                                                    max_features=p["max_features"], random_state=p["random_state"],
                                                    n_jobs=-1))])

def evaluate(y_true, proba, thr=cfg["evaluation"]["threshold"]):
    pred = (proba >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, pred, labels=[0,1]).ravel()
    return {"f1_phishing": f1_score(y_true, pred), "roc_auc": roc_auc_score(y_true, proba),
            "precision": precision_score(y_true, pred), "recall": recall_score(y_true, pred),
            "tn": tn, "fp": fp, "fn": fn, "tp": tp}

## 4. Cross-validation pada development set
Setiap model dilatih pada 4 fold dan diuji pada 1 fold sisanya, diulang lima kali. Threshold tetap 0,5.

In [6]:
rows = []
Xd, yd = X.iloc[dev].reset_index(drop=True), y[dev]
for model_name in ["logistic_regression", "random_forest"]:
    for k, (tr, va) in enumerate(folds, start=1):
        pipe = make_pipeline(model_name)
        t0 = time.perf_counter(); pipe.fit(Xd.iloc[tr], yd[tr]); fit_s = time.perf_counter() - t0
        proba = pipe.predict_proba(Xd.iloc[va])[:, 1]
        m = evaluate(yd[va], proba)
        rows.append({"scenario": "A", "protocol": "random", "model": model_name, "fold": k,
                     "n_features": len(FEATURES_A), "n_train": len(tr), "n_val": len(va),
                     "fit_seconds": round(fit_s, 2), **m})
        print(f"{model_name:20s} fold {k}: F1={m['f1_phishing']:.4f}  AUC={m['roc_auc']:.4f}  ({fit_s:.0f}s)")

per_fold = pd.DataFrame(rows)
per_fold.to_csv(os.path.join(RESULTS_DIR, "metrics_per_fold.csv"), index=False)

/home/zafka/BINDO/Paper/ttki-P1/venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(


logistic_regression  fold 1: F1=0.9991  AUC=1.0000  (9s)


/home/zafka/BINDO/Paper/ttki-P1/venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(


logistic_regression  fold 2: F1=0.9990  AUC=1.0000  (5s)


/home/zafka/BINDO/Paper/ttki-P1/venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(


logistic_regression  fold 3: F1=0.9990  AUC=1.0000  (2s)


/home/zafka/BINDO/Paper/ttki-P1/venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(


logistic_regression  fold 4: F1=0.9990  AUC=1.0000  (2s)


/home/zafka/BINDO/Paper/ttki-P1/venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(


logistic_regression  fold 5: F1=0.9992  AUC=1.0000  (2s)
random_forest        fold 1: F1=0.9998  AUC=1.0000  (16s)
random_forest        fold 2: F1=0.9999  AUC=1.0000  (16s)
random_forest        fold 3: F1=0.9998  AUC=1.0000  (17s)
random_forest        fold 4: F1=0.9999  AUC=1.0000  (17s)
random_forest        fold 5: F1=0.9998  AUC=1.0000  (16s)


## 5. Ringkasan (rata-rata ± simpangan baku antar fold)

In [7]:
metrics = ["f1_phishing", "roc_auc", "precision", "recall"]
summ = per_fold.groupby("model")[metrics].agg(["mean", "std"]).round(4)
summ.to_csv(os.path.join(RESULTS_DIR, "baseline_A_cv_summary.csv"))
for model_name, g in per_fold.groupby("model"):
    print(model_name, " | ".join(f"{m}: {g[m].mean():.4f} ± {g[m].std():.4f}" for m in metrics))
print()
print(per_fold.groupby("model")[["tn","fp","fn","tp"]].sum())

logistic_regression f1_phishing: 0.9990 ± 0.0001 | roc_auc: 1.0000 ± 0.0000 | precision: 0.9993 ± 0.0002 | recall: 0.9988 ± 0.0002
random_forest f1_phishing: 0.9998 ± 0.0001 | roc_auc: 1.0000 ± 0.0000 | precision: 0.9999 ± 0.0001 | recall: 0.9997 ± 0.0002

                         tn  fp   fn     tp
model                                      
logistic_regression  107826  54  100  80656
random_forest        107874   6   21  80735


## 6. Test final (dimatikan)
Sel di bawah baru dijalankan **sekali**, setelah seluruh rancangan (fitur, kebijakan, parameter) dibekukan dan disepakati
dengan Dika dan dosen. Setelah dijalankan, jangan ubah apa pun lalu menjalankannya lagi.

In [8]:
if RUN_FINAL_TEST:
    out = []
    for model_name in ["logistic_regression", "random_forest"]:
        pipe = make_pipeline(model_name).fit(X.iloc[dev], y[dev])
        out.append({"scenario":"A","model":model_name, **evaluate(y[test], pipe.predict_proba(X.iloc[test])[:,1])})
    pd.DataFrame(out).to_csv(os.path.join(RESULTS_DIR, "test_final_A.csv"), index=False)
    print(pd.DataFrame(out).round(4))
else:
    print("Test final belum dijalankan (RUN_FINAL_TEST = False).")

Test final belum dijalankan (RUN_FINAL_TEST = False).


## 7. Cara membaca hasil
- **Skor sangat tinggi pada split acak itu wajar** untuk dataset ini dan belum membuktikan model mampu mendeteksi phishing
  pada domain yang belum pernah dilihat. Itulah yang diuji skenario B dan C.
- Bandingkan **simpangan baku antar fold**: nilai kecil berarti hasil stabil terhadap pembagian data.
- Hasil ini adalah titik acuan A. Jangan menafsirkan perbedaan Logistic Regression dan Random Forest sebagai "model terbaik";
  keduanya hanya pembanding.
- Catat bila ada hal janggal (misalnya skor mendekati sempurna) di bagian temuan, apa adanya.